# Black Carbon EDA & Visualization

Exploratory analysis for merged `model_features` tables.

**Outputs** (under `outputs/eda/`):
- Spatial BC map with traffic-weighted markers
- Midday vehicle composition vs. cleaned BC
- Raw vs. cleaned BC distribution comparison

Core logic lives in `bc_eda.py`. See [README.md](README.md) for pipeline integration.


In [ ]:
%matplotlib inline

from pathlib import Path

from bc_eda import (
    ensure_bc_cleaned,
    load_spatial_data,
    plot_bc_raw_vs_cleaned,
    plot_midday_activity_and_correlations,
    plot_spatial_bc_map,
    prepare_midday_resampled,
)

PACKAGE_DIR = Path(".").resolve()
DATA_DIR = PACKAGE_DIR / "data"
OUTPUT_DIR = PACKAGE_DIR / "outputs" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Configuration

Set `SPATIAL_CSV` and `DAY2_CSV` to your merged `model_features` CSV paths (see [README.md](README.md)).
If `bc_cleaned` is missing, it is derived from `bc` when plots load the data.


In [ ]:
USE_MAP_OVERLAY = True

SPATIAL_CSV = DATA_DIR / "model_features_1s.csv"
DAY2_CSV = DATA_DIR / "model_features_1s_day2.csv"

for label, path in [("Spatial", SPATIAL_CSV), ("Day-2", DAY2_CSV)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} input not found: {path}. See README.md.")

print("Spatial input:", SPATIAL_CSV)
print("Day-2 input:", DAY2_CSV)


## 1. Spatial map: BC concentration vs. traffic

In [ ]:
spatial_df = load_spatial_data(SPATIAL_CSV)
map_name = "traffic_map_overlay.png" if USE_MAP_OVERLAY else "traffic_latlon_plot.png"
plot_spatial_bc_map(
    spatial_df,
    output_path=OUTPUT_DIR / map_name,
    use_map_overlay=USE_MAP_OVERLAY,
)


## 2. Midday traffic composition & BC correlations

In [ ]:
import pandas as pd

day2_df = ensure_bc_cleaned(pd.read_csv(DAY2_CSV))
resampled, t_start, t_end = prepare_midday_resampled(day2_df)
paths = plot_midday_activity_and_correlations(
    resampled,
    target_start=t_start,
    target_end=t_end,
    output_dir=OUTPUT_DIR,
)
print("Saved:", paths)


## 3. Raw vs. cleaned black carbon

In [ ]:
fig, p95, max_cleaned = plot_bc_raw_vs_cleaned(
    day2_df,
    output_path=OUTPUT_DIR / "bc_combined_analysis_grayscale.png",
)
print(f"95th percentile (raw): {p95:.0f}")
print(f"Max bc_cleaned: {max_cleaned:.2f}")
